## IEC Voter Registration Extraction Approach

Before extracting the data, the IEC Voter Registration Statistics page was inspected to understand how the website delivers its data.

The page does not expose the required municipality and ward registration data directly in the initial HTML. The province, municipality, and ward selections are handled through the IEC's ASP.NET Web Forms interface.

The browser's network activity was therefore inspected to identify what happens when selections are made. This showed that selecting a province triggers an asynchronous POST request to the same IEC page, carrying ASP.NET form-state fields such as `__VIEWSTATE`, `__EVENTVALIDATION`, the selected province, and the event target.

Based on this observation, we will reproduce the website's normal selection process programmatically rather than trying to invent or rely on an undocumented API.

### Extraction flow

`IEC page → KwaZulu-Natal → Municipality → Ward → Registration statistics`

The extraction will:

1. Start an HTTP session with the IEC website.
2. Load the initial page and obtain the required ASP.NET state fields.
3. Select **KwaZulu-Natal** programmatically.
4. Retrieve the available KwaZulu-Natal municipalities.
5. Select each municipality and retrieve its wards.
6. Retrieve the registration statistics available for each ward.
7. Repeat this across the required **2011–2026** period where the IEC source provides the corresponding data.
8. Verify the extracted structure and records.
9. Save the original extracted results to `data/raw/`.

This notebook is limited to **data extraction and verification**. Cleaning, transformation, feature engineering, analysis, and modelling will be handled later in the project.

In [1]:
# We are extracting voter registration statistics
# for KwaZulu-Natal only. from2011 to 2026
#
# The IEC website uses an ASP.NET form, so we will
# reproduce the same province → municipality → ward
# selection process programmatically.
#
# Raw extracted data will be saved in:
# data/raw/
#
# No cleaning or processing is done in this notebook.

from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# IEC source
IEC_URL = (
    "https://www.elections.org.za/pw/StatsData/"
    "Voter-Registration-Statistics"
)

# Extraction scope
TARGET_PROVINCE = "KwaZulu-Natal"
KZN_PROVINCE_ID = "4"

# Historical period requested
START_YEAR = 2011
CURRENT_YEAR = 2026

print("IEC extraction setup complete.")
print("Source:", IEC_URL)
print("Province:", TARGET_PROVINCE)
print("Province ID:", KZN_PROVINCE_ID)
print("Period:", START_YEAR, "to", CURRENT_YEAR)
print("Raw directory:", RAW_DIR)

IEC extraction setup complete.
Source: https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics
Province: KwaZulu-Natal
Province ID: 4
Period: 2011 to 2026
Raw directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw


In [2]:
#2.
# We first open the IEC page and keep a session active.
# The session is needed because the IEC uses ASP.NET
# state and cookies when moving between selections.

iec_session = requests.Session()

initial_response = iec_session.get(
    IEC_URL,
    timeout=30
)

print("HTTP status:", initial_response.status_code)
print("Content type:", initial_response.headers.get("Content-Type"))
print("Response size:", len(initial_response.content), "bytes")

assert initial_response.status_code == 200, (
    "IEC voter registration page could not be loaded."
)

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

print("IEC source loaded successfully.")
print("Session established successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131362 bytes
IEC source loaded successfully.
Session established successfully.


In [3]:
#3.
# The IEC uses ASP.NET Web Forms.
# We need the hidden state fields generated by
# the page before we can reproduce the browser's
# province-selection request.
#
# These values are extracted dynamically because
# they can change between sessions.

form_state = {}

for field in initial_soup.select(
    "input[type='hidden'][name]"
):
    name = field.get("name")
    value = field.get("value", "")

    form_state[name] = value

print("Hidden form fields found:", len(form_state))

required_fields = [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]

for field in required_fields:
    assert field in form_state, (
        f"Required ASP.NET field missing: {field}"
    )

print("Required ASP.NET form state found.")

print("\nRequired fields:")
for field in required_fields:
    print("-", field)

Hidden form fields found: 3
Required ASP.NET form state found.

Required fields:
- __VIEWSTATE
- __VIEWSTATEGENERATOR
- __EVENTVALIDATION


In [6]:
#4.
# Lets select the province we want to focus on
# The IEC uses ASP.NET AJAX for the province dropdown.
# Our earlier request reached the server but returned
# ER-500, so we now reproduce the browser request more
# closely, including the AJAX headers.

province_post_data = form_state.copy()

province_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlProvinces",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlProvinces",

    "__EVENTARGUMENT":
        "",

    "__LASTFOCUS":
        "",

    "__SCROLLPOSITIONX":
        "0",

    "__SCROLLPOSITIONY":
        "0",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "-1",

    "__ASYNCPOST":
        "true"
})

province_headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "X-MicrosoftAjax": "Delta=true",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": IEC_URL,
    "Origin": "https://www.elections.org.za"
}

province_response = iec_session.post(
    IEC_URL,
    data=province_post_data,
    headers=province_headers,
    timeout=30
)

print("HTTP status:", province_response.status_code)
print("Content type:", province_response.headers.get("Content-Type"))
print("Response size:", len(province_response.content), "bytes")

print("\nResponse preview:")
print(province_response.text[:300])

HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 129533 bytes

Response preview:
1|#||4|112264|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padd


In [9]:
#5. 
# ============================================
# CELL 5 — VERIFY KZN PROVINCE RESPONSE
# ============================================
#
# The IEC returned an ASP.NET AJAX updatePanel response.
# We extract the updated HTML and check that the
# municipality dropdown was populated for KwaZulu-Natal.

response_text = province_response.text

assert "updatePanel|MainContent_MainUpdatePanel|" in response_text, (
    "IEC did not return the expected updatePanel response."
)

# Extract the HTML contained in the updatePanel response
panel_marker = "updatePanel|MainContent_MainUpdatePanel|"

panel_start = response_text.find(panel_marker) + len(panel_marker)
panel_html = response_text[panel_start:]

# Parse the returned HTML
province_soup = BeautifulSoup(
    panel_html,
    "html.parser"
)

# Find the municipality dropdown
municipality_select = province_soup.select_one(
    "select[name='ctl00$MainContent$ddlMunicipalities']"
)

assert municipality_select is not None, (
    "Municipality dropdown was not found in the KZN response."
)

# Extract municipality options
municipality_options = []

for option in municipality_select.find_all("option"):
    value = option.get("value", "").strip()
    text = option.get_text(strip=True)

    if value and value != "-1":
        municipality_options.append({
            "municipality_id": value,
            "municipality": text
        })

print("KZN province response verified.")
print("Municipalities found:", len(municipality_options))

print("\nFirst municipalities:")
for municipality in municipality_options[:10]:
    print(
        municipality["municipality_id"],
        "→",
        municipality["municipality"]
    )

KZN province response verified.
Municipalities found: 44

First municipalities:
4005 → ETH - eThekwini
4403 → KZN212 - uMdoni
4404 → KZN213 - uMzumbe
4405 → KZN214 - uMuziwabantu
4407 → KZN216 - Ray Nkonyeni
4409 → KZN221 - uMshwathi
4410 → KZN222 - uMngeni
4411 → KZN223 - Mpofana
4412 → KZN224 - iMpendle
4413 → KZN225 - Msunduzi


In [13]:
#6.
# The KZN AJAX request succeeded, but its response
# does not contain the hidden ASP.NET fields.
#
# We inspect the response structure before constructing
# the municipality request.

response_text = province_response.text

print("Response length:", len(response_text))

print("\n--- RESPONSE CONTROL RECORDS ---")

for marker in [
    "updatePanel|",
    "asyncPostBackControlIDs|",
    "postBackControlIDs|",
    "updatePanelIDs|",
    "panelsToRefreshIDs|",
    "formAction|",
    "pageTitle|",
    "scriptBlock|"
]:
    position = response_text.find(marker)

    if position >= 0:
        print(
            f"\n{marker}"
            f"\n{response_text[position:position + 500]}"
        )
    else:
        print(f"\n{marker} NOT FOUND")

Response length: 129533

--- RESPONSE CONTROL RECORDS ---

updatePanel|
updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_text display-4

asyncPostBackControlIDs|
asyncPostBackControlIDs|||0|postBackControlIDs|||62|updatePanelIDs||tctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|0|childUpdatePanelIDs|||61|panelsToRefreshIDs||ctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|2|asyncPostBackTimeout||90|31|formAction||./Voter-Registration-Statistics|29|pageTitle||Voter Registration Statistics|

postBackControlIDs|
postBackControlIDs|||62|updatePanelI

In [14]:
#7.
# The successful province response contains the
# updated municipality dropdown.
#
# We inspect its exact HTML and nearby form controls
# so the next postback matches the IEC page structure.

municipality_select = province_soup.select_one(
    "select[name='ctl00$MainContent$ddlMunicipalities']"
)

assert municipality_select is not None, (
    "Municipality dropdown not found."
)

print("Municipality dropdown found.")
print("\nExact municipality control:")
print(municipality_select.prettify()[:5000])

Municipality dropdown found.

Exact municipality control:
<select class="form-control col" id="MainContent_ddlMunicipalities" name="ctl00$MainContent$ddlMunicipalities" onchange="javascript:setTimeout('__doPostBack(\'ctl00$MainContent$ddlMunicipalities\',\'\')', 0)">
 <option selected="selected" value="-1">
  All municipalities
 </option>
 <option value="4005">
  ETH - eThekwini
 </option>
 <option value="4403">
  KZN212 - uMdoni
 </option>
 <option value="4404">
  KZN213 - uMzumbe
 </option>
 <option value="4405">
  KZN214 - uMuziwabantu
 </option>
 <option value="4407">
  KZN216 - Ray Nkonyeni
 </option>
 <option value="4409">
  KZN221 - uMshwathi
 </option>
 <option value="4410">
  KZN222 - uMngeni
 </option>
 <option value="4411">
  KZN223 - Mpofana
 </option>
 <option value="4412">
  KZN224 - iMpendle
 </option>
 <option value="4413">
  KZN225 - Msunduzi
 </option>
 <option value="4414">
  KZN226 - Mkhambathini
 </option>
 <option value="4415">
  KZN227 - Richmond
 </option>
 <opt

In [15]:
#8.
# We inspect the actual HTML form used by the IEC.
# This confirms the form action and any fields that
# must accompany the municipality postback.

main_form = initial_soup.find("form")

assert main_form is not None, (
    "IEC form was not found."
)

print("Form method:", main_form.get("method"))
print("Form action:", main_form.get("action"))

print("\nForm ID:")
print(main_form.get("id"))

print("\nForm name:")
print(main_form.get("name"))

print("\nForm controls:")
for control in main_form.select(
    "input[name], select[name], textarea[name]"
):
    name = control.get("name")
    value = control.get("value", "")

    if name in [
        "__VIEWSTATE",
        "__VIEWSTATEGENERATOR",
        "__EVENTVALIDATION",
        "ctl00$ctl13",
        "ctl00$MainContent$ddlProvinces",
        "ctl00$MainContent$ddlMunicipalities"
    ]:
        print(
            name,
            "=", 
            value[:100] if isinstance(value, str) else value
        )

Form method: post
Form action: ./Voter-Registration-Statistics

Form ID:
uxForm

Form name:
None

Form controls:
__VIEWSTATE = PEIwRd94+e7txW0rm4mt20XfIk4vF7bqP51HUS+tgAxcu4ErFRAnziGx17GJxntzt1t2ZwdSZAHWiZz9dVdxZ9C4MMGZ3JqrTq7q
__VIEWSTATEGENERATOR = 1852F751
__EVENTVALIDATION = Um/vu+5BCzUwtTuojLeRK1DTQPVkktot6jXaZxXGXjQJc06JI1CyBaEXt/NezFBo3B5h4S0m2Lnb0IO6wYjDiiNT3UNHChu0YouN
ctl00$MainContent$ddlProvinces = 
ctl00$MainContent$ddlMunicipalities = 


In [22]:
#9.
# The province request returns an ASP.NET AJAX
# delta response. The updated hidden form fields
# are contained in hiddenField records, so we
# extract those values before selecting a municipality.

import re

# Extract hidden fields returned by the province AJAX response
province_state = {}

hidden_field_pattern = re.compile(
    r"\|hiddenField\|([^|]+)\|([^|]*)"
)

for name, value in hidden_field_pattern.findall(
    province_response.text
):
    province_state[name] = value

print("Updated hidden fields found:", len(province_state))

required_fields = [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]

for field in required_fields:
    assert field in province_state, (
        f"Updated ASP.NET field missing: {field}"
    )

print("Updated ASP.NET state extracted successfully.")

# Build municipality request using the UPDATED state
municipality_post_data = province_state.copy()

municipality_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlMunicipalities",

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

municipality_response = iec_session.post(
    IEC_URL,
    data=municipality_post_data,
    headers=province_headers,
    timeout=30
)

print("\nMunicipality request:")
print("HTTP status:", municipality_response.status_code)
print("Content type:", municipality_response.headers.get("Content-Type"))
print("Response size:", len(municipality_response.content), "bytes")

print("\nResponse preview:")
print(municipality_response.text[:500])

Updated hidden fields found: 8
Updated ASP.NET state extracted successfully.

Municipality request:
HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 84124 bytes

Response preview:
1|#||4|63932|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_t


In [23]:
#10. lets verufy the war table 
# The municipality response should contain the
# registered-voter table for its wards.
#
# We verify the table structure before extracting
# individual ward records.

municipality_soup = BeautifulSoup(
    municipality_response.text,
    "html.parser"
)

# Find the ward table
ward_tables = municipality_soup.find_all("table")

print("Tables found:", len(ward_tables))

assert len(ward_tables) > 0, (
    "No tables found in municipality response."
)

# Look for the table containing the expected ward columns
ward_table = None

for table in ward_tables:
    table_text = table.get_text(" ", strip=True)

    if (
        "Ward" in table_text
        and "Voting districts" in table_text
        and "Registered voters" in table_text
    ):
        ward_table = table
        break

assert ward_table is not None, (
    "Ward registration table not found."
)

print("Ward registration table found.")

# Extract table headers
headers = [
    th.get_text(" ", strip=True)
    for th in ward_table.find_all("th")
]

print("\nTable headers:")
for header in headers:
    print("-", header)

Tables found: 1
Ward registration table found.

Table headers:
- Ward
- Voting districts
- Registered voters


In [25]:
#11.
# now that we have confirmed lets extract wards in page 1 as we did find out that is several pages in wards

# We now extract the ward-level registration
# records from the municipality response.
#
# Page 1 contains the first set of wards returned
# by the IEC for the selected municipality.

ward_rows = []

for row in ward_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"])
    ]

    # Keep only rows containing the three expected fields
    if len(cells) == 3 and cells[0].isdigit():
        ward_rows.append(cells)

print("Ward records found on page 1:", len(ward_rows))

assert len(ward_rows) > 0, (
    "No ward records were extracted."
)

print("\nFirst 5 ward records:")

for row in ward_rows[:5]:
    print(row)

Ward records found on page 1: 20

First 5 ward records:
['59500001', '14', '19,964']
['59500002', '21', '21,172']
['59500003', '13', '17,103']
['59500004', '9', '20,987']
['59500005', '5', '14,950']


In [26]:
#12.
#Before touching pagination, let's put these records into a structured raw dataset and verify the values


# We convert the extracted IEC rows into a DataFrame
# while preserving the values returned by the source.
#
# No cleaning or transformation is performed here.

page1_df = pd.DataFrame(
    ward_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Rows:", len(page1_df))
print("Columns:", list(page1_df.columns))

print("\nFirst 5 records:")
print(page1_df.head())

Rows: 20
Columns: ['ward', 'voting_districts', 'registered_voters']

First 5 records:
       ward voting_districts registered_voters
0  59500001               14            19,964
1  59500002               21            21,172
2  59500003               13            17,103
3  59500004                9            20,987
4  59500005                5            14,950


In [28]:
#13.
# ============================================
# CELL #13 — INSPECT PAGINATION CONTROLS
# ============================================
#
# The IEC pager is rendered as ASP.NET controls.
# We inspect the actual links/buttons returned by
# the municipality response instead of assuming an ID.

# Find all links and inputs related to the ward pager
pager_elements = municipality_soup.find_all(
    lambda tag: (
        tag.name in ["a", "input"]
        and (
            "WardDataPager" in str(tag)
            or "uxWardDataPager" in str(tag)
        )
    )
)

print("Pager elements found:", len(pager_elements))

assert len(pager_elements) > 0, (
    "IEC ward pagination controls were not found."
)

print("\nPagination controls:\n")

for element in pager_elements:
    print(
        "TAG:", element.name,
        "| TEXT:", element.get_text(" ", strip=True),
        "| NAME:", element.get("name"),
        "| HREF:", element.get("href")
    )

Pager elements found: 7

Pagination controls:

TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl00$ctl00 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl01 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl02 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl03 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl04 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl01$ctl05 | HREF: None
TAG: input | TEXT:  | NAME: ctl00$MainContent$uxRegisteredVotersWardListView$uxWardDataPager$ctl02$ctl00 | HREF: None


In [29]:
#14. LETS THEN: reproduce the Page 2 click programmatically.
# We reproduce the IEC pager's Page 2 control.
# The ASP.NET state must come from the current
# municipality response.

# Extract the latest ASP.NET state from the
# municipality AJAX response.
municipality_state = {}

for name, value in re.findall(
    r"\|hiddenField\|([^|]+)\|([^|]*)",
    municipality_response.text
):
    municipality_state[name] = value

print("Updated hidden fields found:", len(municipality_state))

assert "__VIEWSTATE" in municipality_state
assert "__EVENTVALIDATION" in municipality_state

# Page 2 control identified from the actual IEC response
page2_control = (
    "ctl00$MainContent$uxRegisteredVotersWardListView"
    "$uxWardDataPager$ctl01$ctl02"
)

page2_post_data = municipality_state.copy()

page2_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        + page2_control,

    "__EVENTTARGET": page2_control,

    "__EVENTARGUMENT": "",
    "__LASTFOCUS": "",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "4005",

    "__ASYNCPOST":
        "true"
})

page2_response = iec_session.post(
    IEC_URL,
    data=page2_post_data,
    headers=province_headers,
    timeout=30
)

print("\nPage 2 request:")
print("HTTP status:", page2_response.status_code)
print("Content type:", page2_response.headers.get("Content-Type"))
print("Response size:", len(page2_response.content), "bytes")

print("\nResponse preview:")
print(page2_response.text[:500])

Updated hidden fields found: 8

Page 2 request:
HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 84085 bytes

Response preview:
1|#||4|63893|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_t


In [30]:
#15.
# We parse the Page 2 response and confirm that
# it contains ward records different from Page 1.

page2_soup = BeautifulSoup(
    page2_response.text,
    "html.parser"
)

# Find the ward table
page2_ward_table = None

for table in page2_soup.find_all("table"):
    table_text = table.get_text(" ", strip=True)

    if (
        "Ward" in table_text
        and "Voting districts" in table_text
        and "Registered voters" in table_text
    ):
        page2_ward_table = table
        break

assert page2_ward_table is not None, (
    "Ward registration table not found on Page 2."
)

# Extract Page 2 ward records
page2_rows = []

for row in page2_ward_table.find_all("tr"):
    cells = [
        cell.get_text(" ", strip=True)
        for cell in row.find_all(["td", "th"])
    ]

    if len(cells) == 3 and cells[0].isdigit():
        page2_rows.append(cells)

print("Page 2 ward records:", len(page2_rows))

assert len(page2_rows) > 0, (
    "No ward records found on Page 2."
)

print("\nFirst 5 Page 2 records:")

for row in page2_rows[:5]:
    print(row)

# Confirm Page 2 is different from Page 1
page1_wards = {row[0] for row in ward_rows}
page2_wards = {row[0] for row in page2_rows}

print("\nPage 1 first ward:", ward_rows[0][0])
print("Page 2 first ward:", page2_rows[0][0])

assert page1_wards.isdisjoint(page2_wards), (
    "Page 2 contains wards already found on Page 1."
)

print("\nPage 2 verification passed.")

Page 2 ward records: 20

First 5 Page 2 records:
['59500041', '5', '17,053']
['59500042', '8', '18,545']
['59500043', '7', '16,185']
['59500044', '7', '16,686']
['59500045', '7', '17,493']

Page 1 first ward: 59500001
Page 2 first ward: 59500041

Page 2 verification passed.


In [32]:
#16. now lets extract page 2 into a Dataframe,

# We convert the verified Page 2 records into
# the same raw structure used for Page 1.

page2_df = pd.DataFrame(
    page2_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Rows:", len(page2_df))
print("Columns:", list(page2_df.columns))

print("\nFirst 5 Page 2 records:")
print(page2_df.head())


Rows: 20
Columns: ['ward', 'voting_districts', 'registered_voters']

First 5 Page 2 records:
       ward voting_districts registered_voters
0  59500041                5            17,053
1  59500042                8            18,545
2  59500043                7            16,185
3  59500044                7            16,686
4  59500045                7            17,493


In [33]:
#17.
# Pages 1 and 2 were verified manually.
# We now repeat the same ASP.NET pagination process
# for Pages 3, 4, and 5.

additional_page_rows = {}

for page_number in range(3, 6):

    print(f"\nRequesting Page {page_number}...")

    # Extract the latest ASP.NET state from the
    # response of the previous page.
    current_state = {}

    for name, value in re.findall(
        r"\|hiddenField\|([^|]+)\|([^|]*)",
        page2_response.text
        if page_number == 3
        else current_page_response.text
    ):
        current_state[name] = value

    assert "__VIEWSTATE" in current_state
    assert "__EVENTVALIDATION" in current_state

    # Page controls:
    # Page 3 = ctl01$ctl03
    # Page 4 = ctl01$ctl04
    # Page 5 = ctl01$ctl05
    page_control = (
        "ctl00$MainContent$uxRegisteredVotersWardListView"
        "$uxWardDataPager$ctl01$ctl0"
        + str(page_number)
    )

    page_post_data = current_state.copy()

    page_post_data.update({
        "ctl00$ctl13":
            "ctl00$MainContent$MainUpdatePanel|"
            + page_control,

        "__EVENTTARGET": page_control,
        "__EVENTARGUMENT": "",
        "__LASTFOCUS": "",

        "ctl00$MainContent$ddlProvinces":
            KZN_PROVINCE_ID,

        "ctl00$MainContent$ddlMunicipalities":
            "4005",

        "__ASYNCPOST":
            "true"
    })

    current_page_response = iec_session.post(
        IEC_URL,
        data=page_post_data,
        headers=province_headers,
        timeout=30
    )

    print(
        "HTTP status:",
        current_page_response.status_code
    )

    print(
        "Response size:",
        len(current_page_response.content),
        "bytes"
    )

    assert current_page_response.status_code == 200
    assert "updatePanel|MainContent_MainUpdatePanel" in (
        current_page_response.text
    )

    # Parse the page
    current_soup = BeautifulSoup(
        current_page_response.text,
        "html.parser"
    )

    current_ward_table = None

    for table in current_soup.find_all("table"):
        table_text = table.get_text(" ", strip=True)

        if (
            "Ward" in table_text
            and "Voting districts" in table_text
            and "Registered voters" in table_text
        ):
            current_ward_table = table
            break

    assert current_ward_table is not None, (
        f"Ward table not found on Page {page_number}."
    )

    # Extract ward rows
    rows = []

    for row in current_ward_table.find_all("tr"):
        cells = [
            cell.get_text(" ", strip=True)
            for cell in row.find_all(["td", "th"])
        ]

        if len(cells) == 3 and cells[0].isdigit():
            rows.append(cells)

    print(
        f"Page {page_number} ward records:",
        len(rows)
    )

    assert len(rows) == 20, (
        f"Expected 20 wards on Page {page_number}, "
        f"found {len(rows)}."
    )

    additional_page_rows[page_number] = rows

print("\nPages 3–5 extraction completed successfully.")


Requesting Page 3...
HTTP status: 200
Response size: 84085 bytes
Page 3 ward records: 20

Requesting Page 4...
HTTP status: 200
Response size: 84087 bytes
Page 4 ward records: 20

Requesting Page 5...
HTTP status: 200
Response size: 72977 bytes
Page 5 ward records: 12


AssertionError: Expected 20 wards on Page 5, found 12.

In [34]:
#18.
#lets investigate and be sure by what is returned by page 5
# The final IEC page may contain fewer records
# than the standard page size.
#
# Page 5 returned 12 wards, so we verify that
# these are valid ward records rather than treating
# the smaller final page as an extraction failure.

page5_rows = rows

print("Page 5 ward records:", len(page5_rows))

assert len(page5_rows) > 0, (
    "No ward records were extracted from Page 5."
)

print("\nPage 5 records:")

for row in page5_rows:
    print(row)

print("\nPage 5 verification passed.")

Page 5 ward records: 12

Page 5 records:
['59500101', '4', '18,890']
['59500102', '5', '16,632']
['59500103', '12', '21,012']
['59500104', '4', '15,719']
['59500105', '26', '16,780']
['59500106', '6', '16,132']
['59500107', '5', '16,071']
['59500108', '10', '15,302']
['59500109', '7', '19,849']
['59500110', '5', '19,858']
['59500111', '11', '15,638']
['59500112', '6', '17,140']

Page 5 verification passed.


In [35]:
#19. 
#We combine the verified ward pages into one
# raw municipality-level dataset.
#
# No cleaning or transformation is performed.
# The values remain exactly as returned by the IEC.

all_ethekwini_rows = (
    ward_rows
    + page2_rows
    + additional_page_rows[3]
    + additional_page_rows[4]
    + page5_rows
)

ethekwini_df = pd.DataFrame(
    all_ethekwini_rows,
    columns=[
        "ward",
        "voting_districts",
        "registered_voters"
    ]
)

print("Total ward records:", len(ethekwini_df))

print(
    "Unique ward records:",
    ethekwini_df["ward"].nunique()
)

print("\nColumns:")
print(list(ethekwini_df.columns))

assert len(ethekwini_df) == 92, (
    f"Expected 92 ward records, "
    f"found {len(ethekwini_df)}."
)

assert ethekwini_df["ward"].nunique() == 92, (
    "Duplicate ward records detected."
)

print("\nFirst 5 wards:")
print(ethekwini_df.head())

print("\nLast 5 wards:")
print(ethekwini_df.tail())

print("\nEThekwini ward extraction verification passed.")

Total ward records: 92
Unique ward records: 92

Columns:
['ward', 'voting_districts', 'registered_voters']

First 5 wards:
       ward voting_districts registered_voters
0  59500001               14            19,964
1  59500002               21            21,172
2  59500003               13            17,103
3  59500004                9            20,987
4  59500005                5            14,950

Last 5 wards:
        ward voting_districts registered_voters
87  59500108               10            15,302
88  59500109                7            19,849
89  59500110                5            19,858
90  59500111               11            15,638
91  59500112                6            17,140

EThekwini ward extraction verification passed.


In [36]:
# okay we have executed a strategy test and veruified it so
# Before we automate the other municipalities, let's save this verified extraction.
#This follows our EXTRACT then VERIFY then SAVE RAW workflow. and what is nice no duplicates detected the logic seems to be promising,

# The eThekwini ward extraction has been verified,
# so we now save the raw IEC response data.


ethekwini_raw_path = (
    RAW_DIR / "iec_kzn_ethekwini_ward_registration_2026.csv"
)

ethekwini_df.to_csv(
    ethekwini_raw_path,
    index=False
)

print("Raw file saved:")
print(ethekwini_raw_path)

print("\nRows saved:", len(ethekwini_df))
print("Columns saved:", list(ethekwini_df.columns))

# Verify that the saved file can be read back
saved_ethekwini_df = pd.read_csv(
    ethekwini_raw_path,
    dtype=str
)

print("\nSaved file verification:")
print("Rows read back:", len(saved_ethekwini_df))
print("Columns read back:", list(saved_ethekwini_df.columns))

assert len(saved_ethekwini_df) == 92
assert list(saved_ethekwini_df.columns) == [
    "ward",
    "voting_districts",
    "registered_voters"
]

print("\nEThekwini raw file verification passed.")

Raw file saved:
C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw\iec_kzn_ethekwini_ward_registration_2026.csv

Rows saved: 92
Columns saved: ['ward', 'voting_districts', 'registered_voters']

Saved file verification:
Rows read back: 92
Columns read back: ['ward', 'voting_districts', 'registered_voters']

EThekwini raw file verification passed.
